In [1]:
import datetime
import numpy as np
import pandas as pd
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import spearmanr
import seaborn as sns
from sklearn.model_selection import train_test_split
import joblib

In [2]:
def performance_report(y_actual, y_predict, y_proba, header):
        
    print('\033[1m' + "\n--------- " + header + " ---------\n" + '\033[0m')
    
    # 1. Overall Accuracy
    acc = accuracy_score(y_actual, y_predict)
    print("1. Overall Accuracy: {}%".format(round(100*acc,1)))

    # 2. Precision
    prec = precision_score(y_actual, y_predict)
    print("2. Precision: {}%".format(round(100*prec,1)))

    # 3. Recall
    rec = recall_score(y_actual, y_predict)
    print("3. Recall: {}%".format(round(100*rec,1)))

    # 4. F1 score
    f1_scr = f1_score(y_actual, y_predict)
    print("4. F1-score: {}%".format(round(100*f1_scr,1)))

    # 5. AUC
    auc_scr = roc_auc_score(y_actual, y_proba)
    print("5. AUC Score: {}".format(round(auc_scr,3)))
    
    # print("\n6. Confusion matrix:\n")
    
    # cm = confusion_matrix(y_actual, y_predict)
    # disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    # fig = plt.figure(figsize = (3,3))
    # axs = fig.subplots(1, 1)
    # disp.plot(ax = axs, values_format = 'd')

    return [round(100*f1_scr,1), round(100*prec,1), round(100*rec,1), round(auc_scr,3)]


In [3]:
# Loading train & test dataset
train_data = pd.read_csv("dataset/train_data.csv")
test_data = pd.read_csv("dataset/test_data.csv")

# Loading models trained on train data
dectree_f1 = joblib.load('models/dectree_f1.pkl')
dectree_reg = joblib.load('models/dectree_reg.pkl')

<p style='font-size:16px'><b> Test dataset </b></p>
<p>

Important highlight related to test dataset would be regarding the categorical features that were converted using target encoding. <br>
In real world scenarios, we may encounter new values of these features e.g product_id that weren't present in the train data set. <br>
In that case the target encoding of these features needs to be estimated as per context, for e,g: <br>
- product_id to pid_new: Can be obtained by assigning it the average pid_new value of the category that it belongs.
- Similarly pincode_new can be calculated throught the average pincode_new value of the city/zone it belongs to.

However, for purpose of this task, since the new values encountered in test dataset are very few as calculated below, we'll continue with the target encoded values that were computed during preprocessing.

</p>


In [5]:
train_pids = set(train_data.product_id)
test_pids = set(test_data.product_id)

train_city = set(train_data.city)
test_city = set(test_data.city)

train_pincode = set(train_data.pincode)
test_pincode = set(test_data.pincode)

train_vertical = set(train_data.vertical)
test_vertical = set(test_data.vertical)

print("1a. # of distinct product_ids in test set: ", len(test_pids))
print("1b. # of new product_ids in test set: ", len(test_pids - train_pids))

print("\n2a. # of distinct cities in test set: ", len(test_city))
print("2b. # of new cities in test set: ", len(test_city - train_city))
print("\n3. # of new pincodes in test set: ", len(test_pincode - train_pincode))
print("4. # of new verticals in test set: ", len(test_vertical - train_vertical))


1a. # of distinct product_ids in test set:  18429
1b. # of new product_ids in test set:  4926

2a. # of distinct cities in test set:  3275
2b. # of new cities in test set:  601

3. # of new pincodes in test set:  137
4. # of new verticals in test set:  0


In [6]:
train_data.drop(['product_id', 'city', 'pincode', 'vertical'], axis = 1, inplace = True)
test_data.drop(['product_id', 'city', 'pincode', 'vertical'], axis = 1, inplace = True)

In [7]:
# Split into X (input) & Y (output)

train_x = train_data.drop('y_label', axis = 1)
train_y = train_data['y_label']

test_x = test_data.drop('y_label', axis = 1)
test_y = test_data['y_label']

In [8]:
# Performance on train & test data (GCV best model)
y_pred_train = dectree_f1.predict(train_x)
y_proba_train = dectree_f1.predict_proba(train_x)[:,-1]
model1_train = performance_report(train_y, y_pred_train, y_proba_train, 'Model 1: train data')

y_pred_test = dectree_f1.predict(test_x)
y_proba_test = dectree_f1.predict_proba(test_x)[:,-1]
model1_test = performance_report(test_y, y_pred_test, y_proba_test, 'Model 1: test data')


--------- Model 1: train data ---------

1. Overall Accuracy: 99.9%
2. Precision: 99.4%
3. Recall: 98.2%
4. F1-score: 98.8%
5. AUC Score: 1.0

--------- Model 1: test data ---------

1. Overall Accuracy: 98.3%
2. Precision: 91.5%
3. Recall: 69.4%
4. F1-score: 78.9%
5. AUC Score: 0.982


In [9]:
# Performance on train & test data (Regularised version)
y_pred_train = dectree_reg.predict(train_x)
y_proba_train = dectree_reg.predict_proba(train_x)[:,-1]
model2_train = performance_report(train_y, y_pred_train, y_proba_train, 'Model 2: train data')

y_pred_test = dectree_reg.predict(test_x)
y_proba_test = dectree_reg.predict_proba(test_x)[:,-1]
model2_test = performance_report(test_y, y_pred_test, y_proba_test, 'Model 2: test data')


--------- Model 2: train data ---------

1. Overall Accuracy: 98.5%
2. Precision: 95.6%
3. Recall: 70.9%
4. F1-score: 81.4%
5. AUC Score: 0.996

--------- Model 2: test data ---------

1. Overall Accuracy: 98.0%
2. Precision: 90.6%
3. Recall: 63.5%
4. F1-score: 74.7%
5. AUC Score: 0.966


<p style='font-size:16px'><b> Model performance summary </b></p>
<p>

The performance of both the models based on previously defined metrics has been populated below. <br>
Both the models perform almost similarly on test data, but the Model 2 offers better generalisation due to regularisation. <br>
In a real world scenario, a high precision model is more likely to be useful for this application.
For e.g: By deploying this model we would be able to reduce fraud orders by 62.5% while causing minimal disruption in normal orders.

</p>


In [11]:
metrics = ['F1 Score', 'Precision', 'Recall', 'AUC Score']

perf_summary = pd.DataFrame({'model1_train': model1_train, 'model1_test': model1_test, \
                'model2_train': model2_train, 'model2_test': model2_test}, \
                index=pd.Index(metrics, name='CustomIndex'))

perf_summary

,model1_train,model1_test,model2_train,model2_test
CustomIndex,,,,
F1 Score,98.8,78.900,81.400,74.700
Precision,99.4,91.500,95.600,90.600
Recall,98.2,69.400,70.900,63.500
AUC Score,1.0,0.982,0.996,0.966


<p style='font-size:16px'><b> Improvements </b></p>
<p>

Mentioning the following steps that would be beneficial in improving the results of this problem:
- Outlier detection while data preprocessing for cleaner data.
- Other ensemble techniques, like XGBoost (Boosting), or cascading models to sequentially filter out normal transactions.
- Visualising the decision tree path that led to maximum fraud orders.
- Additional features like user age, gender, affluency or FK plus customer, etc.
                                                              
</p>